# P1-C: bilateral factored diagonal regularizer

This notebook constructs the exact article link, the two independently oriented core-regularizer relations, and the ordered candidate $T_{1,-}R_1Z_1^{-1}$. It deliberately leaves common-algebra membership, a single Mellin-PDO representation, and the Schur symbol unproved.

In [1]:
from pathlib import Path
import sys

import sympy as sp

repo_root = Path.cwd()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from symbolic_operator_calculus import (  # noqa: E402
    AssumptionContext, AuxiliaryOperator, CompactIdeal, ComplexDomain,
    CoreRegularizerClaimStatus, CoreRegularizerProperty, DiagonalLinkIdentity,
    DiagonalLinkStatus, DiagonalOperator, ExactIdentity, ExactIdentityScope,
    LinearCombination, MellinCoreOperator, MellinExpression,
    MellinPseudodifferentialOperator, MellinSymbol, MellinSymbolDependency,
    MellinSymbolDomain, MultiplicationOperator, OperatorAtom, Product,
    RegularizerMellinRepresentation, RegularizerOperator,
    RegularizerRepresentationStatus, SingularSet, Term, TransportedMellinCore,
    WeightedLpSpace, certify_factored_two_sided_diagonal_regularizer,
    insert_factored_regularizer_in_schur, make_mellin_core_regularizer_certificate,
    mellin_frequency, output_spatial_variable,
)
from symbolic_operator_calculus.diagonal_regularizer import (  # noqa: E402
    InvertibleAuxiliaryOperator, InvertibleMultiplicationOperator,
)

## Weighted space and typed Mellin symbols

The scalar functions below are formal placeholders for the article's displayed $n_1$ and $r_{1,1}$; their analytic regularizer property is supplied only through the cited theorem evidence.

In [2]:
gamma = sp.Symbol('gamma', positive=True)
kappa = sp.Symbol('kappa', real=True)
c1 = sp.Symbol('c_1', positive=True)
x = sp.Symbol('x', positive=True)
lam = sp.Symbol('lambda', real=True)
assumptions = AssumptionContext((gamma > 0, kappa > 0, kappa < 1))
space = WeightedLpSpace(
    label='X=L^p(R_+,w_delta)', p=sp.Integer(2), weight_exponent=sp.Integer(0),
    assumptions=assumptions, source='article scalar weighted realization',
)

frequency = mellin_frequency(lam)
radial = output_spatial_variable(x)
symbol_domain = MellinSymbolDomain(
    frequency=frequency, complex_domain=ComplexDomain.real_line(lam),
    spatial_variables=(radial,), assumption_context=assumptions,
    singular_set=SingularSet(), description='P1-C formal symbol domain',
    evidence=('article Mellin realization',),
)

def mellin_pdo(name, scalar):
    expression = MellinExpression(
        expression=scalar, domain=symbol_domain, variables=(frequency, radial),
        evidence=(f'article formula placeholder for {name}',), description=name,
    )
    symbol = MellinSymbol(
        expression, MellinSymbolDependency.SPACE_FREQUENCY, description=name,
    )
    return MellinPseudodifferentialOperator(
        atom=OperatorAtom(f'Op_{name}'), symbol=symbol, domain=space, codomain=space,
        symbol_class='tilde-E', source=f'article Mellin operator {name}',
        radial_scaling_stable=True, stability_evidence=('symbol-class theorem',),
        evidence=('boundedness under the stated Fredholm hypotheses',),
    )

Op_n1 = mellin_pdo('n1', sp.Function('n_1')(x, lam))
Op_r11 = mellin_pdo('r11', sp.Function('r_11')(x, lam))
space

WeightedLpSpace(label='X=L^p(R_+,w_delta)', p=2, weight_exponent=0, assumptions=AssumptionContext(assumptions=(kappa > 0, kappa < 1), consistency_status=<ConsistencyStatus.CONSISTENT: 'consistent'>), source='article scalar weighted realization', norm_convention=<WeightNormConvention.MULTIPLICATIVE_POWER_WEIGHT: 'integral |x**delta f(x)|**p dx'>, measure='dx on (0,infinity)')

## Exact diagonal interfaces

$T_{1,-}$ remains the noncommutative sum $U_1^{-1}P^+ + P^-$, and $Z_1$ remains multiplication by the article's $\zeta_1$.

In [3]:
def multiplier(name, scalar, source):
    return MultiplicationOperator(
        atom=OperatorAtom(name), symbol=scalar, radial_variable=x,
        domain=space, codomain=space, source=source, evidence=(source,),
    )

U1_inverse = multiplier('U1_inverse', sp.Integer(1), 'typed U1 inverse component')
P_plus = multiplier('P_plus', sp.Integer(1), 'typed Cauchy P+ component')
P_minus = multiplier('P_minus', sp.Integer(1), 'typed Cauchy P- component')
T1_minus_raw = AuxiliaryOperator(
    atom=OperatorAtom('T1_minus'),
    expression=LinearCombination((
        Term(1, Product((U1_inverse.atom, P_plus.atom))), Term(1, P_minus.atom),
    )),
    components=(U1_inverse, P_plus, P_minus), domain=space, codomain=space,
    source='article section 05:515-529',
    evidence=('T1_minus=U1_inverse P_plus+P_minus',),
)
theta_inverse = multiplier(
    'Op_theta1_inverse', sp.Integer(1), 'article inverse formula component',
)
T1_inverse_raw = AuxiliaryOperator(
    atom=OperatorAtom('T1_minus_inverse'),
    expression=LinearCombination((
        Term(1, theta_inverse.atom), Term(1, Product((P_plus.atom, P_minus.atom))),
    )),
    components=(theta_inverse, P_plus, P_minus), domain=space, codomain=space,
    source='article section 05:770-778', evidence=('exact inverse theorem',),
)
T1_minus = InvertibleAuxiliaryOperator(
    operator=T1_minus_raw, inverse=T1_inverse_raw,
    source='article section 05:532-782', evidence=('exact invertibility',),
)

zeta1 = gamma**(-kappa) * (gamma*x-sp.I*c1)/(x-sp.I*c1)
Z1_raw = multiplier('Z1', zeta1, 'article section 05:339-408')
Z1_inverse = multiplier('Z1_inverse', 1/zeta1, 'exact reciprocal of zeta1')
Z1 = InvertibleMultiplicationOperator(
    operator=Z1_raw, inverse=Z1_inverse, source='article exact Z1 inverse',
    evidence=('pointwise reciprocal identities',),
    nonvanishing_evidence=('continuous extension and nonzero endpoint values',),
)

A11 = DiagonalOperator(
    atom=OperatorAtom('A11'), domain=space, codomain=space,
    source='article section 06:27-42',
    evidence=('A11=Vtilde_1 P_plus+G1 P_minus',),
)
print('A11:', A11.atom.name)
print('T1_minus:', T1_minus.operator.exact_definition.right)
print('Z1:', Z1.operator.symbol)
print('Z1_inverse:', Z1.inverse.symbol)

A11: A11
T1_minus: LinearCombination(terms=(Term(coefficient=1, product=Product(factors=(OperatorAtom(name='U1_inverse', kind='operator'), OperatorAtom(name='P_plus', kind='operator')))), Term(coefficient=1, product=Product(factors=(OperatorAtom(name='P_minus', kind='operator'),)))))
Z1: (-I*c_1 + gamma*x)/(gamma**kappa*(-I*c_1 + x))
Z1_inverse: gamma**kappa*(-I*c_1 + x)/(-I*c_1 + gamma*x)


## Mellin core, transported regularizer, and exact link

The notebook declares $H_1:=N_1$ only as a P1-C alias; the article itself uses $N_1$.

In [4]:
Phi = OperatorAtom('Phi', kind='space_isomorphism')
Phi_inverse = OperatorAtom('Phi_inverse', kind='space_isomorphism')
H1 = MellinCoreOperator(
    atom=OperatorAtom('H1'), mellin_operator=Op_n1, phi=Phi,
    phi_inverse=Phi_inverse, source='article sections 05:792-900 and 06:109-132',
    evidence=('Phi N1 Phi_inverse=Op(n1)',), alias_of='article operator N1',
)
R1_abstract = RegularizerOperator(
    target=H1.atom, operator=OperatorAtom('R1', kind='formal_regularizer'),
)
R1_representation = RegularizerMellinRepresentation(
    regularizer=R1_abstract, mellin_operator=Op_r11,
    status=RegularizerRepresentationStatus.CERTIFIED_EXACT,
    hypotheses=assumptions.assumptions, source='article section 06:161-168',
    evidence=('R1=Phi_inverse Op(r11) Phi',), space=space,
)
R1_transport = TransportedMellinCore(
    representation=R1_representation, phi_inverse=Phi_inverse, phi=Phi,
    source='article exact R1 transport', evidence=('section 06:161-168',),
)
compact_ideal = CompactIdeal(
    label='K(X)', space=space, bilateral=True, source='article section 06:10-12',
    evidence=('compact operators form a bilateral ideal',),
)
R1 = make_mellin_core_regularizer_certificate(
    mellin_operator=H1, mellin_regularizer=R1_transport,
    property=CoreRegularizerProperty.TWO_SIDED_REGULARIZER,
    claim_status=CoreRegularizerClaimStatus.CERTIFIED,
    compact_ideal=compact_ideal, assumptions=assumptions,
    source='article section 06:134-171',
    evidence=('both Op(n1)Op(r11) and Op(r11)Op(n1) are identity mod K',),
)
link_left = Product((Z1.inverse_atom, A11.atom, T1_minus.atom))
link_right = Product((H1.atom,))
link = DiagonalLinkIdentity(
    diagonal_operator=A11, left_interface=T1_minus, mellin_operator=H1,
    right_interface=Z1, right_interface_inverse=Z1_inverse,
    left_expression=link_left, right_expression=link_right,
    relation=ExactIdentity(
        link_left, link_right, evidence=('article exact link',),
        scope=ExactIdentityScope.OPERATORIAL,
    ),
    compact_ideal=None, domain=space, codomain=space, assumptions=assumptions,
    evidence=('A11=Z1 Ahat11 and N1=Ahat11 T1_minus',),
    source='article sections 05:429-507 and 05:792-900',
    status=DiagonalLinkStatus.EXACT,
)
print('H1 (P1-C alias of N1):', H1.atom.name)
print('R1:', R1.atom.name, '/', R1.property.value)
print('link:', link.left_expression, '=', link.right_expression)

H1 (P1-C alias of N1): H1
R1: R1 / TWO_SIDED_REGULARIZER
link: Product(factors=(OperatorAtom(name='Z1_inverse', kind='operator'), OperatorAtom(name='A11', kind='operator'), OperatorAtom(name='T1_minus', kind='operator'))) = Product(factors=(OperatorAtom(name='H1', kind='operator'),))


## Bilateral certification and independent traces

In [5]:
result = certify_factored_two_sided_diagonal_regularizer(
    diagonal_operator=A11, left_interface=T1_minus, mellin_operator=H1,
    mellin_regularizer=R1, right_interface=Z1,
    right_interface_inverse=Z1_inverse, link_identity=link,
    assumptions=assumptions,
)
print('candidate:', result.candidate_regularizer.ast_product)
print('left product:', result.left_product)
for step in result.left_reduction_trace:
    print(' ', step.step, '=>', step.output_expression, '/', type(step.relation).__name__)
print('right product:', result.right_product)
for step in result.right_reduction_trace:
    print(' ', step.step, '=>', step.output_expression, '/', type(step.relation).__name__)
print('bilateral status:', result.status.value)

candidate: Product(factors=(OperatorAtom(name='T1_minus', kind='operator'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Z1_inverse', kind='operator')))
left product: Product(factors=(OperatorAtom(name='A11', kind='operator'), OperatorAtom(name='T1_minus', kind='operator'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Z1_inverse', kind='operator')))
  left-1-link => Product(factors=(OperatorAtom(name='Z1', kind='operator'), OperatorAtom(name='H1', kind='operator'), OperatorAtom(name='R1', kind='formal_regularizer'), OperatorAtom(name='Z1_inverse', kind='operator'))) / ExactIdentity
  left-2-core-regularizer => Product(factors=(OperatorAtom(name='Z1', kind='operator'), OperatorAtom(name='I', kind='identity'), OperatorAtom(name='Z1_inverse', kind='operator'))) / ModCompactEquivalence
  left-3-exact-cancellation => Product(factors=(OperatorAtom(name='I', kind='identity'),)) / ExactIdentity
right product: Product(factors=(OperatorAtom(nam

## Limited structural Schur insertion

The certified center is inserted as one opaque atom. No cutoff, algebra-membership, Mellin-PDO, or symbol claim is generated.

In [6]:
L21 = multiplier('L21', sp.Integer(1), 'left Schur exterior placeholder')
R12 = multiplier('R12', sp.Integer(1), 'right Schur exterior placeholder')
insertion = insert_factored_regularizer_in_schur(
    left_exterior=L21, central_regularizer=result, right_exterior=R12,
)
print('Schur insertion:', insertion.schur_insertion.value)
print('structural word:', insertion.expression)
print('expanded:', insertion.expanded_expression)
print('common algebra:', insertion.common_algebra_membership.value)
print('single Mellin PDO:', insertion.single_mellin_pdo_representation.value)
print('Schur symbol:', insertion.schur_symbol.value)

Schur insertion: STRUCTURALLY_VALID
structural word: Product(factors=(OperatorAtom(name='L21', kind='operator'), OperatorAtom(name='A11_factored_regularizer', kind='factored_regularizer'), OperatorAtom(name='R12', kind='operator')))
expanded: None
common algebra: UNPROVED
single Mellin PDO: UNPROVED
Schur symbol: NOT_CERTIFIED


In [7]:
print('open obligations:')
for obligation in result.proof_obligations:
    print(' -', obligation.key)
print('LaTeX:')
print(result.latex)

open obligations:
 - full_regularizer_algebra_membership
 - single_mellin_pdo_representation
 - cutoff_replacement_mod_compacts
 - wh_mellin_wh_sandwich_membership
 - fredholm_algebra_membership
 - schur_correction_symbol
LaTeX:
B_1=\mathcal T_{1,-}\mathcal R_1Z_1^{-1},\quad \mathcal A_{1,1}B_1\simeq I,\quad B_1\mathcal A_{1,1}\simeq I\quad\text{CERTIFIED_TWO_SIDED_FACTORED_REGULARIZER}
